# Build an eval set & score a prompt

**Session 3 · Track A · local Ollama**

The pivot: measure your classifier instead of eyeballing it.

In [ ]:
import sys; sys.path.append('..')  # so `utils` and `eval` import from the repo root
from eval import load_cases, run_eval, print_report, exact
sys.path.append('../02_techniques')
from importlib import import_module


Load the labelled cases and score the few-shot classifier from Session 2.

In [ ]:
# re-define the classifier here (or import it) so this notebook is self-contained
from utils import ask
FEWSHOT = """Classify the sentiment as positive, negative, or neutral.
Reply with ONE lowercase word and nothing else.

Text: "I love this" -> positive
Text: "crashed twice" -> negative
Text: "It is a phone" -> neutral
"""
def classify(text):
    out = ask(FEWSHOT + f'Text: "{text}" -> ').strip().lower()
    for lab in ["positive","negative","neutral"]:
        if lab in out: return lab
    return out

cases = load_cases("../eval/datasets/sentiment.jsonl")
report = run_eval(cases, classify, scorer=exact)
print_report(report)

### Now iterate
Change ONE thing (add an example, tighten the instruction), re-run, and compare the number.

### Worked example

Iterate on the prompt with the eval set held fixed: add two examples aimed at the failures, then compare before/after accuracy.


In [ ]:
# Worked example: change ONE thing, keep the eval fixed, compare the number
BASE = FEWSHOT   # from the cell above

IMPROVED = FEWSHOT + (
    'Text: "Great, another update that breaks everything" -> negative\n'   # sarcasm
    'Text: "Works, but the setup was painful" -> negative\n'               # mixed -> negative
)

def make_classifier(fewshot):
    def classify(text):
        out = ask(fewshot + f'Text: "{text}" -> ').strip().lower()
        return next((l for l in ["positive", "negative", "neutral"] if l in out), out)
    return classify

before = run_eval(cases, make_classifier(BASE),     scorer=exact)
after  = run_eval(cases, make_classifier(IMPROVED), scorer=exact)
print(f"before: {before['accuracy']:.0%}   after: {after['accuracy']:.0%}\n")
print_report(after)


## Your turn - vary the example

1. Look at the FAIL lines, then add few-shot examples aimed exactly at those cases.
2. Re-score. Did overall accuracy go up, or did you just trade one failure for another?
3. Record before/after accuracy in your commit message and PR.


In [ ]:
# Your variation here - copy the worked example above and change ONE thing, then re-run
